# Ex 1

In [1]:
class LFSR:
    def __init__(self, feedback_coefficients, initial_state):
        self.coefficients = feedback_coefficients
        self.state = initial_state.copy()
        self.L = len(initial_state)
        
        if len(feedback_coefficients) != self.L:
            raise ValueError("Number of coefficients must equal the length of the state")
        
        if not all(c in [0, 1] for c in self.coefficients):
            raise ValueError("The coefficinets must be 0 or 1")
        if not all(s in [0, 1] for s in self.state):
            raise ValueError("The  initial state only containts 0 or 1")
    
    def step(self):
        output = self.state[0]
        
        # s_(t+L) = sum(c_i * s_(t+L-i)) mod 2
        feedback_bit = 0
        for i in range(self.L):
            feedback_bit ^= self.coefficients[self.L - 1 - i] * self.state[i]
        
        self.state = self.state[1:] + [feedback_bit]
        
        return output
    
    def generate_sequence(self, length):
        sequence = []
        for _ in range(length):
            sequence.append(self.step())
        return sequence
    
    def reset(self, new_state=None):
        if new_state is None:
            raise ValueError("You need to provide a new state for the reset")
        if len(new_state) != self.L:
            raise ValueError(f"The state needs to be of length {self.L}")
        self.state = new_state.copy()
    
    def get_state(self):
        return self.state.copy()

In [2]:
# Table 1. Successive states of the LFSR with feedback coefficients 
# (c1, c2, c3, c4) = (0, 0, 1, 1) and with initial state
# (s0, s1, s2, s3) = (1, 0, 1, 1)

lfsr_example = LFSR(
    feedback_coefficients=[0, 0, 1, 1],
    initial_state=[1, 0, 1, 1]
)

print(f"{'t':<5} {'s_t':<5} {'s_t+1':<8} {'s_t+2':<8} {'s_t+3':<8}")
print("-" * 40)

for t in range(16):
    output = lfsr_example.step()
    state = lfsr_example.get_state()
    print(f"{t:<5} {output:<5} {state[0]:<8} {state[1]:<8} {state[2]:<8}")

t     s_t   s_t+1    s_t+2    s_t+3   
----------------------------------------
0     1     0        1        1       
1     0     1        1        1       
2     1     1        1        1       
3     1     1        1        0       
4     1     1        0        0       
5     1     0        0        0       
6     0     0        0        1       
7     0     0        1        0       
8     0     1        0        0       
9     1     0        0        1       
10    0     0        1        1       
11    0     1        1        0       
12    1     1        0        1       
13    1     0        1        0       
14    0     1        0        1       
15    1     0        1        1       


In [3]:
lfsr_example.reset([1, 0, 1, 1])
sequence = lfsr_example.generate_sequence(30)
print(f"First 30 bits: {''.join(map(str, sequence))}")

First 30 bits: 101111000100110101111000100110


# Ex 2

In [4]:
# a)
from Crypto.Cipher import AES

key = b'O cheie oarecare'
data=b'testtesttesttesttesttesttesttesttesttesttesttest'

cipher = AES.new(key, AES.MODE_ECB)
print("Obtinem:")
cipher.encrypt(data)

Obtinem:


b'\x88\x10\x86\xe2\xf3\xaai)\x9fz\xcb\xf0h4\xa4\xec\x88\x10\x86\xe2\xf3\xaai)\x9fz\xcb\xf0h4\xa4\xec\x88\x10\x86\xe2\xf3\xaai)\x9fz\xcb\xf0h4\xa4\xec'

b. Modul de operare folosit este ECB (Electronic Codebook)
   Observație: ECB criptează fiecare bloc în mod independent.
   Dacă avem blocuri identice în plaintext, vor produce blocuri identice în ciphertext. În cazul nostru, 'test' se repetă, deci vom vedea pattern-uri repetitive în output.


c. Nu este recomandată folosirea modului ECB, deoarece:
   - nu se pot ascunde repetițiile din date
   - blocurile identice corespund unui ciphertext identic, deci vulnerabilitate la atacurile de analiză

In [5]:
# d)
print(len(key))
print(f"in biti: {len(key) * 8}")

16
in biti: 128


In [6]:
# e)
from Crypto.Util.Padding import pad

key_e = b'O cheie oarecare'
data_short = b'test'

cipher_e = AES.new(key_e, AES.MODE_ECB)
# padding pentru a ajunge la multiplu de 16
data_padded = pad(data_short, AES.block_size)
encrypted_e = cipher_e.encrypt(data_padded)

print(f"Data init: {data_short}")
print(f"Data padded: {data_padded}")
print(f"Encrypted: {encrypted_e.hex()}\n")

Data init: b'test'
Data padded: b'test\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c'
Encrypted: e8aaa5da09dec3bfa4d4235230b901b7



In [7]:
# f)
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad

key_f = b'O cheie oarecare'
data_f = b'testtesttesttesttesttesttesttesttesttesttesttest'

# iv aleator
iv = get_random_bytes(16)
cipher_cbc = AES.new(key_f, AES.MODE_CBC, iv)

# padding
data_padded_f = pad(data_f, AES.block_size)
encrypted_cbc = cipher_cbc.encrypt(data_padded_f)

print(f"CBC - cipher block chaining")
# nu mai apar structuri repetitive
print(encrypted_cbc)

CBC - cipher block chaining
b'S\x9aW\x87\x8a\xa6\x1a\xc5\xd7&T\xce\xe5\xcem\x9bv{\x92\x0c\xd4\xe7%\xc5=6W\xb4\xe0s\xedH2\xa6:\x9d\xe6\x1c\x80\xe1\xb8\x94\xd3\xeb{\xf0\xca\xb1\xfd\x9d#$\xe4\x88\xfdu\x8a\x14\xd7R\toX\xd7'


# Ex 3

In [8]:
from Crypto.Cipher import DES

ciphertext = b"G\xfd\xdfpd\xa5\xc9'C\xe2\xf0\x84)\xef\xeb\xf9"
plaintext = b"Provocare MitM!!"

print("Plaintext:", plaintext)
print("Ciphertext:", ciphertext.hex())

Plaintext: b'Provocare MitM!!'
Ciphertext: 47fddf7064a5c92743e2f08429efebf9


In [9]:
# tabel pentru criptarile lui key1
encryption_table_key1 = {}

# valori posibile (0-15)
for nibble in range(16):
    byte_val = (nibble << 4) | 0x0
    key1_candidate = bytes([byte_val, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00])
    cipher1 = DES.new(key1_candidate, DES.MODE_ECB)
    intermediate = cipher1.encrypt(plaintext)
    encryption_table_key1[intermediate] = nibble

print(f"Tabela key1 generata: {len(encryption_table_key1)} intrari")

Tabela key1 generata: 16 intrari


In [10]:
# cautare in tabela de decriptare cu key2
found_keys = []

for nibble in range(16):
    byte_val = (nibble << 4) | 0x0
    key2_candidate = bytes([byte_val, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00])
    cipher2 = DES.new(key2_candidate, DES.MODE_ECB)
    
    # decriptare ciphertext cu key2
    try:
        intermediate_decrypt = cipher2.decrypt(ciphertext)
        
        # verificam daca rezultatul apartine tabelei key1
        if intermediate_decrypt in encryption_table_key1:
            key1_nibble = encryption_table_key1[intermediate_decrypt]
            found_keys.append((key1_nibble, nibble))
            print(f"Rezultat: key1[0] = 0x{key1_nibble:01x}0, key2[0] = 0x{nibble:01x}0")
    except:
        pass

Rezultat: key1[0] = 0x80, key2[0] = 0xe0


In [11]:
# verificare perechi
for key1_nibble, key2_nibble in found_keys:
    byte1 = (key1_nibble << 4) | 0x0
    byte2 = (key2_nibble << 4) | 0x0
    
    key1 = bytes([byte1, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00])
    key2 = bytes([byte2, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00])
    
    cipher1 = DES.new(key1, DES.MODE_ECB)
    cipher2 = DES.new(key2, DES.MODE_ECB)
    
    test_ciphertext = cipher2.encrypt(cipher1.encrypt(plaintext))
    
    if test_ciphertext == ciphertext:
        print(f"key1 = {key1.hex()}")
        print(f"key2 = {key2.hex()}")

key1 = 8000000000000000
key2 = e000000000000000


Cate chei au fost testate in total?
  - Pentru key1: 16 chei (0-F)
  - Pentru key2: 16 chei (0-F)
  - Total: 16 + 16 = 32 chei

Cate criptari/decriptari s-au facut?
  - Criptari cu key1: 16
  - Decriptari cu key2: 16
  - Total: 32 operatii